# Clasificación — Predicción de Dirección del Oro

**Autor**: Juan

**Objetivo**: Predecir si el oro sube o baja al día siguiente.

**Pipeline**: Este notebook recibe el CSV generado por Gema (`gold-features.csv`) y aplica modelos de clasificación.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from src.classification import get_models, evaluate, overfitting_check, backtest, feature_importance

NO_FEATURE_COLS = ['Date', 'target_binary', 'target_multiclass']

def create_modeling_data(df):
    X = df.drop(columns=NO_FEATURE_COLS)
    y_bin = df['target_binary']
    y_multi = df['target_multiclass']
    return X, y_bin, y_multi

def temporal_train_test_split(X, y, test_size=0.2):
    split = int(len(X) * (1 - test_size))
    return X.iloc[:split].copy(), X.iloc[split:].copy(), y.iloc[:split].copy(), y.iloc[split:].copy()

def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    return X_train_s, X_test_s, scaler

sns.set_style('whitegrid')
%matplotlib inline

## 1. Carga de datos

Cargamos el CSV que Gema genera con su pipeline: `preprocessing.py` → `feature_engineering.py` → `targets.py`

In [ ]:
df = pd.read_csv('../data/processed/gold-features.csv', parse_dates=['Date'])
print(f'Dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Columnas: {df.columns.tolist()}')
df.head()

### 1.1 Distribución de clases

In [ ]:
print('=== Binaria (sube/baja) ===')
print(df['target_binary'].value_counts(normalize=True))
print()
print('=== Multiclase (comprar/mantener/vender) ===')
print(df['target_multiclass'].value_counts(normalize=True))

## 2. Preparación para modelado

Usamos `pipeline.py` de Gema para separar features/targets, hacer split temporal y escalar.

In [ ]:
X, y_bin, y_multi = create_modeling_data(df)
X_train, X_test, y_train_bin, y_test_bin = temporal_train_test_split(X, y_bin)
_, _, y_train_multi, y_test_multi = temporal_train_test_split(X, y_multi)
X_train_s, X_test_s, scaler = scale_features(X_train, X_test)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test:  {X_test.shape[0]} muestras')
print(f'Features: {X_train.shape[1]}')

## 3. Modelos

In [ ]:
models = get_models()

### 3.1 Clasificación binaria

In [ ]:
results_bin = evaluate(models, X_train_s, X_test_s, y_train_bin, y_test_bin)
results_bin

### 3.2 Clasificación multiclase

In [ ]:
results_multi = evaluate(models, X_train_s, X_test_s, y_train_multi, y_test_multi, average='weighted')
results_multi

### 3.3 Backtesting

In [ ]:
returns_test = df['gold_return_next_day'].iloc[X_test.index[0]:].values
bt = backtest(models, X_train_s, X_test_s, y_train_bin, y_test_bin, returns_test)
bt

## 4. Detección de overfitting

In [ ]:
of = overfitting_check(models, X_train_s, X_test_s, y_train_bin, y_test_bin)
of

## 5. Importancia de features (Random Forest)

In [ ]:
rf = models['Random Forest']
rf.fit(X_train_s, y_train_bin)
imp = feature_importance(rf, X.columns)

fig, ax = plt.subplots(figsize=(10, 6))
imp.head(15).plot(kind='barh', ax=ax)
ax.set_title('Top 15 Feature Importance (Random Forest)')
ax.set_xlabel('Importancia')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Conclusiones

| Aspecto | Resultado |
|---------|-----------|
| **Mejor modelo binario** | (completar) |
| **Mejor modelo multiclase** | (completar) |
| **Overfitting detectado** | (completar) |
| **Estrategia vs Buy & Hold** | (completar) |
| **Features más importantes** | (completar) |